In [1]:
import json
import pandas as pd

In [2]:
with open('big_east.json') as f:
    big_east = json.load(f)

In [3]:
def round_assign(x):
    if x['POSTSEASON'] == 5 : return [1,1,1,1,1]
    elif x['POSTSEASON'] == 4 : return [1,1,1,1,0]
    elif x['POSTSEASON'] == 3 : return [1,1,1,0,0]
    elif x['POSTSEASON'] == 2 : return [1,1,0,0,0]
    elif x['POSTSEASON'] == 1 : return [1,0,0,0,0]
    else: return [0,0,0,0,0]

In [4]:
def assign_dummy(df):
    dummy_df = []
    for x in range(0,len(df)):
        dummy_df.append(round_assign(df.iloc[x]))
    dummy_df = pd.DataFrame(dummy_df)
    dummy_df = dummy_df.rename(columns={0: "R1", 1: "R2", 2: "R3",3: "R4", 4:"R5"})
    return dummy_df

In [5]:
df_past_R1 = []

In [6]:
for x in big_east.keys():
    year = (int(x)-2000)

    # Get Tuple from json file
    globals()["be" + '%s' % str(year)] = big_east[x]

    # Get information from that json file year
    globals()["be" + '%s' % str(year) + '%s' % "_seed"] = pd.DataFrame([{"TEAM": x[0], "SEED": globals()["be" + '%s' % str(year)].index(x) + 1, "POSTSEASON": x[1]} for x in globals()["be" + '%s' % str(year)]])
    
    # Uploads the data from the saved csv
    globals()["df" + '%s' % str(year)] = pd.read_csv(f'../data/full/cbb{year}.csv')
    globals()["df" + '%s' % str(year)] = globals()["df" + '%s' % str(year)].apply(lambda x: x.str.strip() if x.dtype == "object" else x)
    if 'POSTSEASON' in globals()["df" + '%s' % str(year)].columns:
        globals()["df" + '%s' % str(year)] = globals()["df" + '%s' % str(year)].drop(columns=['POSTSEASON'])    

    # joins the csv data with the json file
    globals()["df_be" + '%s' % str(year)] = globals()["df" + '%s' % str(year)].drop(columns = ['SEED']).merge(globals()["be" + '%s' % str(year) + '%s' % "_seed"], on='TEAM', how='inner').sort_values('SEED').reset_index(drop=True)

    globals()["df_be" + '%s' % str(year)]['WIN_PER'] = globals()["df_be" + '%s' % str(year)]['W']/globals()["df_be" + '%s' % str(year)]['G']
    globals()["df_be" + '%s' % str(year)]['POWER'] = 1
    globals()["df_be" + '%s' % str(year)]['YEAR'] = int(x)

    globals()["df_be" + '%s' % str(year)] =globals()["df_be" + '%s' % str(year)].join(assign_dummy(globals()["df_be" + '%s' % str(year)]))
    df_past_R1.append(globals()["df_be" + '%s' % str(year)])

In [7]:
new_columns = ['TEAM',
 'CONF',
 'POSTSEASON',
 'G',
 'W',
 "WIN_PER",
 'ADJOE',
 'ADJDE',
 'BARTHAG',
 'EFG_O',
 'EFG_D',
 'TOR',
 'TORD',
 'ORB',
 'DRB',
 'FTR',
 'FTRD',
 '2P_O',
 '2P_D',
 '3P_O',
 '3P_D',
 'ADJ_T',
 'WAB',
 'SEED',
 "POWER",
 'YEAR']

In [8]:
train_columns = [
 'G',
 'W',
 "WIN_PER",
 'ADJOE',
 'ADJDE',
 'BARTHAG',
 'EFG_O',
 'EFG_D',
 'TOR',
 'TORD',
 'ORB',
 'DRB',
 'FTR',
 'FTRD',
 '2P_O',
 '2P_D',
 '3P_O',
 '3P_D',
 'ADJ_T',
 'WAB',
 'SEED'
 ]

In [9]:
def get_upset_differences(a,b):
    """
    Takes two rows from one of the region databases and finds the difference between the two columns 

    a: higher seed
    b: lower seed'

    output: list of db difference rows
    """
    
    listA = []
    for x in range(3,24): # may need 25
        diff = a.iloc[x] - b.iloc[x]
        listA.append(diff)
    return listA

In [10]:
def get_target_variable(df, nxt_round_num, rnd_name):
    """
    This function used to get the target variable attributes from the correct column
    The list is reversed because we want to see if the lower seed wins

    df: dataframe used
    nxt_round_num: number of teams in the next round
    rnd_name: the column name of the dummy round variable of the next round

    output: list of target variable attributes (will be its own column)
    """
    
    listB = list(df[rnd_name])
    listB.reverse()
    listB = listB[0:nxt_round_num]
    return listB

In [11]:
def create_training_record(df, matchup_num, reseed = True):
    """
    This function takes the dataframe, and creates a new dataset of differences that can be trained
    
    df: dataframe of the round being processed
    matchup_num: the number of matchups
    """
    
    listDF = []
    for y in range(0,matchup_num):
        test_upsetH = df.iloc[y]
        test_upsetL = df.iloc[-(y+1)]
        listDF.append(get_upset_differences(test_upsetH,test_upsetL))
    if reseed:
        return pd.DataFrame(listDF, columns = train_columns).sort_values(by = ["SEED"])
    else:
        return pd.DataFrame(listDF, columns = train_columns).sort_values(by = ["SEED"])

In [12]:
def creation(df, next_rounds):
    """
    This function takes the current round's dataframe and the next round string name to use the previous functions to build the training df

    df: current round's dataframe
    next_rounds
    """
    
    y = pd.DataFrame(columns = train_columns)
    asd = []
    nxt_round_num = int(len(df)/2)
    upset= get_target_variable(df,nxt_round_num, next_rounds)
    for x in range(0,nxt_round_num):
        h = df.iloc[x]
        l = df.iloc[(-x-1)]
        print(f"{h['TEAM']} vs {l['TEAM']}")
        if upset[x] == 1:
            print(f"Winner: {l['TEAM']}")
            asd.append(l)
        if upset[x] == 0:
            print(f"Winner: {h['TEAM']}")
            asd.append(h)
    next_df = pd.DataFrame(asd)
    y = pd.concat([y, create_training_record(df,nxt_round_num)], ignore_index=True)
    y["TRAIN"] = upset
    y.TRAIN = y.TRAIN.astype(int)
    return y, next_df

In [13]:
def create_train(a, next_round):
    df = pd.DataFrame(columns = train_columns)
    df_next = []
    for x in a:
        train, next_df = creation(x,next_round)
        df = pd.concat([df,train], ignore_index = True)
        df_next.append(next_df)
    return df, df_next


In [14]:
df = pd.DataFrame(columns = train_columns)
df_next = []

In [15]:
round_name = ["R2", "R3", "R4", "R5"]

In [16]:
def bye_split(df):
    team_cnt = len(df)
    x = 0
    while 2**x < team_cnt:
        binary_rnd = 2**x 
        x += 1
    
    second_round = binary_rnd + (binary_rnd - team_cnt)
    
    return df.iloc[:second_round],df.iloc[second_round:]

In [53]:
df_past_R1

[         TEAM CONF   G   W  ADJOE  ADJDE  BARTHAG  EFG_O  EFG_D   TOR  ...  \
 0   Villanova   BE  36  33  121.9   91.1   0.9663   55.3   45.5  16.3  ...   
 1  Georgetown   BE  33  22  112.6   93.5   0.8945   51.3   47.1  19.2  ...   
 2      Butler   BE  34  23  108.5   88.6   0.9115   48.4   46.7  17.0  ...   
 3  Providence   BE  34  22  111.6   93.7   0.8821   48.4   48.2  18.1  ...   
 4  St. John's   BE  32  20  110.6   94.7   0.8569   49.3   46.4  15.7  ...   
 5      Xavier   BE  37  23  115.7   95.1   0.9049   53.3   50.0  18.1  ...   
 6      DePaul   BE  32  12  107.5  103.8   0.5972   50.7   50.8  19.8  ...   
 7  Seton Hall   BE  31  16  107.7   98.6   0.7353   48.2   47.7  18.7  ...   
 8   Marquette   BE  32  13  104.0   95.6   0.7237   49.8   49.2  19.3  ...   
 9   Creighton   BE  33  14  109.6   99.8   0.7450   49.4   50.5  18.4  ...   
 
    SEED  POSTSEASON   WIN_PER  POWER  YEAR  R1  R2  R3  R4  R5  
 0     1           5  0.916667      1  2015   1   1   1   1   1

In [17]:
train_master = pd.DataFrame(columns = train_columns)
for y in df_past_R1[:-1]:
    for x in range(len(round_name)):
        print(x)
        if x == 0:
            if len(bye_split(y)[1]) == 0:
                print('e')
                pass
            else:
                train_df, first_round = bye_split(y)
                holder = create_train([first_round], round_name[x])
                df_past_R2 = pd.concat([train_df, holder[1][0]], ignore_index=True)
                train_master = pd.concat([train_master,holder[0]], ignore_index = True)

                continue
    
        holder = create_train([globals()["df_past_R" + '%s' % (x+1)]], round_name[x])
        globals()["df_past_R" + '%s' % (x+2)] = holder[1][0]
        train_master = pd.concat([train_master, holder[0]], ignore_index=True)

0
DePaul vs Creighton
Winner: Creighton
Seton Hall vs Marquette
Winner: Marquette
1
Villanova vs Marquette
Winner: Villanova
Georgetown vs Creighton
Winner: Georgetown
Butler vs Xavier
Winner: Xavier
Providence vs St. John's
Winner: Providence
2
Villanova vs Providence
Winner: Villanova
Georgetown vs Xavier
Winner: Xavier
3
Villanova vs Xavier
Winner: Villanova
0
Marquette vs St. John's
Winner: Marquette
Georgetown vs DePaul
Winner: Georgetown
1
Villanova vs Georgetown
Winner: Villanova
Xavier vs Marquette
Winner: Xavier
Seton Hall vs Creighton
Winner: Seton Hall
Providence vs Butler
Winner: Providence
2
Villanova vs Providence
Winner: Villanova
Xavier vs Seton Hall
Winner: Seton Hall
3
Villanova vs Seton Hall
Winner: Seton Hall
0
Xavier vs DePaul
Winner: Xavier
St. John's vs Georgetown
Winner: St. John's
1
Villanova vs St. John's
Winner: Villanova
Butler vs Xavier
Winner: Xavier
Creighton vs Providence
Winner: Creighton
Marquette vs Seton Hall
Winner: Seton Hall
2
Villanova vs Seton H

/var/folders/lg/999sdvtx3kn70j4mnf9pnf980000gn/T/ipykernel_1709/2454364818.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  y = pd.concat([y, create_training_record(df,nxt_round_num)], ignore_index=True)
/var/folders/lg/999sdvtx3kn70j4mnf9pnf980000gn/T/ipykernel_1709/949454954.py:6: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df,train], ignore_index = True)
/var/folders/lg/999sdvtx3kn70j4mnf9pnf980000gn/T/ipykernel_1709/2916054488.py:13: FutureWarning: The behavior of DataF

In [18]:
train_master

,G,W,WIN_PER,ADJOE,ADJDE,BARTHAG,EFG_O,EFG_D,TOR,TORD,...,FTR,FTRD,2P_O,2P_D,3P_O,3P_D,ADJ_T,WAB,SEED,TRAIN
0,-2,-2.1,4.0,-0.1478,1.3,0.3,1.4,1.7,-3.9,7.4,...,1.6,5.8,0.6,-6.8,4.6,-1.6,-3,-1,-0.049242,1.0
1,3,3.7,3.0,0.0116,-1.6,-1.5,-0.6,-3.0,5.7,-3.2,...,-2.8,2.6,0.6,-6.0,1.9,2.4,-1,-1,0.109879,1.0
2,2,1.0,-1.0,0.0252,-0.9,1.8,2.4,0.2,7.6,-6.4,...,0.7,2.2,-3.2,0.7,-1.3,1.4,-1,1,0.022059,0.0
3,0,-7.2,-6.5,0.0066,-4.9,-3.3,-1.1,0.7,3.4,-2.0,...,-7.4,-1.8,0.5,-3.9,-1.6,2.2,-3,-2,0.054849,0.0
4,8,3.0,-6.3,0.1495,1.9,-3.4,0.8,3.1,3.0,2.9,...,2.9,-3.8,0.7,-1.8,1.5,9.5,-8,1,0.242424,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
90,11,9.6,-5.5,0.2674,3.6,-4.4,-0.2,-3.9,-0.5,-1.5,...,7.1,-5.1,0.1,-2.1,-0.2,10.4,0,-8,2,0.0
91,16,-4.0,-16.7,0.1891,-4.5,-4.2,-0.9,10.0,11.8,0.9,...,-1.4,-5.5,-6.9,-1.2,2.3,13.9,0,-8,3,1.0
92,1,-2.4,-2.5,0.0063,1.1,-0.9,0.9,-4.1,-9.1,-0.6,...,3.4,0.4,-0.3,-2.4,3.3,0.7,0,-1,1,0.0
93,8,-3.5,-7.0,0.0405,-2.8,-4.0,1.7,0.0,7.9,-1.2,...,-3.7,-5.3,-2.5,-1.2,2.2,4.2,0,-4,2,0.0


In [19]:
def reset_positive_seeds(df):
    train_neg = df[df["SEED"]<=0]
    train_pos = df[df["SEED"]>0]
    train_pos = - train_pos
    train_pos["TRAIN"] = train_pos["TRAIN"] + 1
    return pd.concat([train_pos, train_neg]).reset_index(drop = True)

In [20]:
train_master= reset_positive_seeds(train_master)


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler 


In [22]:
scaler = StandardScaler()
scaler.fit(train_master.drop(columns = ["TRAIN"]))

def scale(df):
    m = pd.DataFrame(scaler.transform(df), columns = df.columns)
    return m

In [23]:
def formatStuff(df):
    y = df["TRAIN"]
    x = scale(df.drop(columns = ["TRAIN"]))
    return (x,y)

In [24]:
years = [globals()["df_be" + str(int(year) - 2000)] for year in big_east.keys()]

In [25]:
df = pd.concat(years, ignore_index = True)

In [26]:
x_train, y_train = formatStuff(train_master)

In [27]:
import sklearn.tree as tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

In [28]:
#MLP Classifier
iterations = 1000
alpha = 3

# Random Forest Classifier
est = 6
depth = 5

# KNN
neighbors = 3

# Decision Tree
tree_depth = 9

random_state = 69

In [46]:
model =MLPClassifier(max_iter= iterations, alpha= alpha, random_state = random_state)

# KNeighborsClassifier(n_neighbors=neighbors), 
# DecisionTreeClassifier(min_samples_leaf = 5, max_depth = tree_depth), 
# RandomForestClassifier(n_estimators=est, max_depth = depth, random_state = random_state), 
# MLPClassifier(max_iter= iterations, alpha= alpha, random_state = random_state), 
# LogisticRegression(random_state=random_state, C =1), 
# GaussianNB(), 
# SVC(random_state = random_state, C = 1)

model.fit(x_train, y_train)
print("Accuracy on training set: {:.3f}".format(model.score(x_train, y_train)))

Accuracy on training set: 0.958


NEW

In [47]:
# test df
test_df = pd.DataFrame(columns = train_columns)
y_pred = []
matchup_list = []

# compacted implementation for rounds and finals
def play_one_match(h, l):
    """Return (winner_row, holder_df, matchup_str, prediction_int)."""
    holder = pd.DataFrame([get_upset_differences(h, l)], columns=train_columns)
    # ensure holder is oriented so that lower SEED corresponds to label 1 as before
    if holder.iloc[0]["SEED"] > 0:
        holder = -holder
        h, l = l, h
    scaled = scale(holder)
    pred = int(model.predict(scaled)[0])
    winner = h if pred == 0 else l
    matchup = f"{h['TEAM']} vs. {l['TEAM']}"
    return winner, holder, matchup, pred


In [48]:
df_headers = ['TEAM',
 'CONF',
 'POSTSEASON',
 'G',
 'W',
 'WIN_PER',
 'ADJOE',
 'ADJDE',
 'BARTHAG',
 'EFG_O',
 'EFG_D',
 'TOR',
 'TORD',
 'ORB',
 'DRB',
 'FTR',
 'FTRD',
 '2P_O',
 '2P_D',
 '3P_O',
 '3P_D',
 'ADJ_T',
 'WAB',
 'SEED',
 'POWER']

In [49]:
r1 = pd.DataFrame(columns = df_headers)
r2 = pd.DataFrame(columns = df_headers)
r3 = pd.DataFrame(columns = df_headers)
r4 = pd.DataFrame(columns = df_headers)
r5 = pd.DataFrame(columns = df_headers)

In [50]:
def run_rounds(start_df, n_rounds):
    """
    Run n_rounds knockout rounds starting from start_df.
    Returns list_of_rounds where list_of_rounds[0] == start_df,
    and subsequent entries are the winners of each round.
    Side-effects: appends to test_df, y_pred, matchup_list and prints results.
    """
    rounds = [start_df.reset_index(drop=True)]
    for _ in range(n_rounds):
        cur = rounds[-1]
        winners = []
        for i in range(len(cur) // 2):
            h = cur.iloc[i]
            l = cur.iloc[-i-1]
            winner, holder, matchup, pred = play_one_match(h, l)
            winners.append(winner)
            # update global tracking structures
            globals()['test_df'] = pd.concat([globals()['test_df'], holder], ignore_index=True)
            globals()['y_pred'].append(pred)
            globals()['matchup_list'].append(matchup)
            print(f"{h['SEED']} {h['TEAM']}  vs.  {l['SEED']} {l['TEAM']}")
            print("Winner:", winner['SEED'], winner['TEAM'])
            print(' ')
        rounds.append(pd.DataFrame(winners, columns=df_headers))
        print("_" * 30)
    return rounds


In [51]:
test_df26, r1_test = bye_split(df_be26[df_headers])

In [52]:
round1 = run_rounds(r1_test, n_rounds=1)  # r64 -> r32 -> s16 -> e8 -> f4

r1 = pd.concat([test_df26, round1[1]], ignore_index=True)

rounds = run_rounds(r1, n_rounds=3)

# rounds is a list: [r64, r32, s16, e8, f4]
r2 = pd.concat([r2, rounds[1]], ignore_index=True)
r3 = pd.concat([r3, rounds[2]], ignore_index=True)
r4 = pd.concat([r4, rounds[3]], ignore_index=True)

6 DePaul  vs.  11 Xavier
Winner: 11 Xavier
 
7 Butler  vs.  10 Georgetown
Winner: 7 Butler
 
8 Providence  vs.  9 Marquette
Winner: 8 Providence
 
______________________________
1 St. John's  vs.  8 Providence
Winner: 8 Providence
 
2 Connecticut  vs.  7 Butler
Winner: 2 Connecticut
 
3 Villanova  vs.  11 Xavier
Winner: 3 Villanova
 
4 Seton Hall  vs.  5 Creighton
Winner: 5 Creighton
 
______________________________
8 Providence  vs.  5 Creighton
Winner: 8 Providence
 
2 Connecticut  vs.  3 Villanova
Winner: 2 Connecticut
 
______________________________
8 Providence  vs.  2 Connecticut
Winner: 2 Connecticut
 
______________________________


/var/folders/lg/999sdvtx3kn70j4mnf9pnf980000gn/T/ipykernel_1709/1321605409.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  globals()['test_df'] = pd.concat([globals()['test_df'], holder], ignore_index=True)
/var/folders/lg/999sdvtx3kn70j4mnf9pnf980000gn/T/ipykernel_1709/3640770612.py:8: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  r2 = pd.concat([r2, rounds[1]], ignore_index=True)
/var/folders/lg/999sdvtx3kn70j4mnf9pnf980000gn/T/ipykernel_1709/3640770612.py:9: FutureWarning: The behavior o